In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

In [ ]:
CHISCO_ROOT = Path("/projects/EEG-foundation-model/RECH202/speech_raw_datasets/ds005170")

assert CHISCO_ROOT.exists(), f"Root not found: {CHISCO_ROOT}"
print("Dataset root:", CHISCO_ROOT)

## 1. Dataset Overview

**CHISCO** (Chinese Imagined Speech Corpus) is a BCI dataset where participants imagined saying Chinese sentences while EEG was recorded.

- **Format**: BIDS, EDF files
- **Subjects**: sub-01 to sub-05
- **Sessions**: 5–6 sessions per subject
- **Task**: Imagined speech (Chinese sentences)
- **Classes**: 39 semantic topic categories
- **Reference**: https://github.com/zhangzihan-is-good/Chisco

In [ ]:
subjects = sorted(CHISCO_ROOT.glob("sub-*"))
print(f"Number of subjects: {len(subjects)}")
print("Subjects:", [s.name for s in subjects])

In [ ]:
rows = []
for sub_dir in subjects:
    for ses_dir in sorted(sub_dir.glob("ses-*")):
        edf_files = sorted((ses_dir / "eeg").glob("*.edf"))
        for f in edf_files:
            m = re.search(r"run-(\d+)", f.name)
            run_num = int(m.group(1)) if m else None
            rows.append({
                "subject": sub_dir.name,
                "session": ses_dir.name,
                "run": run_num,
                "file": f.name,
                "size_MB": f.stat().st_size / 1024**2,
                "path": str(f),
            })

df_inventory = pd.DataFrame(rows)
print(f"Total EDF files: {len(df_inventory)}")
df_inventory.head(10)

In [ ]:
df_inventory.groupby(["subject", "session"]).size().reset_index(name="n_runs")

In [ ]:
df_inventory.groupby("subject").agg(
    n_sessions=("session", "nunique"),
    n_runs=("file", "count"),
    total_size_MB=("size_MB", "sum"),
).reset_index()

## 2. EDF File Structure — Single File Inspection

In [ ]:
SAMPLE_FILE = CHISCO_ROOT / "sub-01/ses-01/eeg/sub-01_ses-01_task-imagine_run-01_eeg.edf"

raw = mne.io.read_raw_edf(SAMPLE_FILE, preload=False, verbose=False)

print(f"File: {SAMPLE_FILE.name}")
print(f"n_channels: {len(raw.ch_names)}")
print(f"sfreq: {raw.info['sfreq']} Hz")
print(f"duration: {raw.n_times / raw.info['sfreq']:.1f} s  ({raw.n_times / raw.info['sfreq'] / 60:.1f} min)")
print(f"n_annotations: {len(raw.annotations)}")

In [ ]:
ch_types = raw.get_channel_types()
type_counter = Counter(ch_types)
print("Channel type counts:", dict(type_counter))
print()

eeg_chs = [ch for ch, t in zip(raw.ch_names, ch_types) if t == "eeg"]
other_chs = [ch for ch, t in zip(raw.ch_names, ch_types) if t != "eeg"]

print(f"EEG channels ({len(eeg_chs)}):")
print(eeg_chs)
print()
print(f"Non-EEG channels ({len(other_chs)}): {other_chs}")

In [ ]:
unique_desc = sorted(set(raw.annotations.description))
print("Unique annotation descriptions:", unique_desc)
print("n annotations:", len(raw.annotations))
print()
print("First 10 onsets (seconds):")
print(raw.annotations.onset[:10])
print()
onsets = raw.annotations.onset
diffs = np.diff(onsets)
print(f"Inter-trial interval — mean: {diffs.mean():.3f}s, std: {diffs.std():.3f}s")
print(f"  min: {diffs.min():.3f}s, max: {diffs.max():.3f}s")

## 3. Trial Count Across All Files

In [ ]:
trial_rows = []

for _, row in df_inventory.iterrows():
    try:
        r = mne.io.read_raw_edf(row["path"], preload=False, verbose=False)
        trial_rows.append({
            "subject": row["subject"],
            "session": row["session"],
            "run": row["run"],
            "n_trials": len(r.annotations),
            "sfreq": r.info["sfreq"],
            "duration_s": r.n_times / r.info["sfreq"],
        })
    except Exception as e:
        trial_rows.append({
            "subject": row["subject"],
            "session": row["session"],
            "run": row["run"],
            "n_trials": None,
            "error": repr(e),
        })

df_trials = pd.DataFrame(trial_rows)
print(f"Files processed: {len(df_trials)}")
df_trials.head(10)

In [ ]:
df_trials.groupby("subject").agg(
    n_runs=("run", "count"),
    total_trials=("n_trials", "sum"),
    sfreq=("sfreq", "first"),
).reset_index()

In [ ]:
trial_counts = df_trials["n_trials"].dropna().astype(int)
print("n_trials distribution:")
print(trial_counts.value_counts().sort_index())
print(f"\nTotal trials across all subjects: {trial_counts.sum()}")

## 4. Text Data — Labels and Categories

In [ ]:
with open(CHISCO_ROOT / "json/classnumber.json") as f:
    class_map = json.load(f)  # {"0": "自然和天气", ...}

print(f"Number of classes: {len(class_map)}")
print("\nAll categories:")
for k, v in sorted(class_map.items(), key=lambda x: int(x[0])):
    print(f"  {k:>2}: {v}")

In [ ]:
with open(CHISCO_ROOT / "json/textmaps.json") as f:
    text_map = json.load(f)  # {"sentence": class_id, ...}

print(f"Total unique sentences in textmaps: {len(text_map)}")
print("\nSample (sentence: class_id):")
for sent, cls_id in list(text_map.items())[:5]:
    print(f"  {sent!r} -> {cls_id} ({class_map[str(cls_id)]})")

In [ ]:
xlsx_files = sorted(
    CHISCO_ROOT.glob("textdataset/split_data_*.xlsx"),
    key=lambda p: int(re.search(r"(\d+)", p.stem).group(1))
)

print(f"Number of split_data files: {len(xlsx_files)}")

dfs = []
for f in xlsx_files:
    run_num = int(re.search(r"(\d+)", f.stem).group(1))
    df_tmp = pd.read_excel(f)
    df_tmp.columns = ["sentence", "category"]
    df_tmp["run"] = run_num
    dfs.append(df_tmp)

df_text = pd.concat(dfs, ignore_index=True)
print(f"Total rows (trials): {len(df_text)}")
df_text.head(10)

In [ ]:
df_text["class_id"] = df_text["sentence"].map(text_map)

print(f"Sentences with mapped class_id: {df_text['class_id'].notna().sum()} / {len(df_text)}")
print(f"Unique categories (str): {df_text['category'].nunique()}")
print(f"Unique class_ids (int): {df_text['class_id'].nunique()}")

In [ ]:
cat_counts = df_text["category"].value_counts()

fig, ax = plt.subplots(figsize=(12, 8))
cat_counts.sort_values().plot(kind="barh", ax=ax)
ax.set_xlabel("Number of trials")
ax.set_title("CHISCO — Trial count per semantic category (all runs)")
plt.tight_layout()
plt.show()

print("\nTop 10 categories:")
print(cat_counts.head(10))

In [ ]:
run_sizes = df_text.groupby("run").size()
print("Trials per run distribution:")
print(run_sizes.value_counts().sort_index())
print(f"\nMin: {run_sizes.min()}, Max: {run_sizes.max()}, Mean: {run_sizes.mean():.1f}")

## 5. EEG Signal Inspection

In [ ]:
raw = mne.io.read_raw_edf(SAMPLE_FILE, preload=True, verbose=False)

eeg_idx = [i for i, t in enumerate(raw.get_channel_types()) if t == "eeg"]
eeg_names = [raw.ch_names[i] for i in eeg_idx]
data = raw.get_data(picks=eeg_names)  # (n_eeg_ch, n_samples)

print(f"EEG data shape: {data.shape}")
print(f"Min:  {data.min() * 1e6:.2f} µV")
print(f"Max:  {data.max() * 1e6:.2f} µV")
print(f"Mean: {data.mean() * 1e6:.4f} µV")
print(f"Std:  {data.std() * 1e6:.2f} µV")

In [ ]:
ch_stats = pd.DataFrame({
    "channel": eeg_names,
    "mean_uV": data.mean(axis=1) * 1e6,
    "std_uV": data.std(axis=1) * 1e6,
    "min_uV": data.min(axis=1) * 1e6,
    "max_uV": data.max(axis=1) * 1e6,
})

ch_stats.describe()

In [ ]:
print("Channels with std > 200 µV (potential bad channels):")
bad = ch_stats[ch_stats["std_uV"] > 200][["channel", "std_uV"]]
print(bad.to_string() if not bad.empty else "None")

In [ ]:
sfreq = raw.info["sfreq"]
TRIAL_DURATION_S = 10.0

trial_onset_s = raw.annotations.onset[0]
t_start = int(trial_onset_s * sfreq)
t_end = int((trial_onset_s + TRIAL_DURATION_S) * sfreq)

n_plot_ch = 8
plot_ch_names = eeg_names[:n_plot_ch]
plot_data = raw.get_data(picks=plot_ch_names)[:, t_start:t_end] * 1e6

times = np.arange(plot_data.shape[1]) / sfreq

fig, axes = plt.subplots(n_plot_ch, 1, figsize=(14, 2 * n_plot_ch), sharex=True)
for i, (ax, ch) in enumerate(zip(axes, plot_ch_names)):
    ax.plot(times, plot_data[i], lw=0.8)
    ax.set_ylabel(ch, fontsize=8)
    ax.set_ylim(-200, 200)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"CHISCO — First trial ({TRIAL_DURATION_S:.0f}s) — {SAMPLE_FILE.stem}", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
raw_eeg = raw.copy().pick("eeg")
raw_eeg.compute_psd(fmax=60).plot(show=True)

## 6. Signal Statistics Across All Files (sub-01)

In [ ]:
sub01_files = sorted((CHISCO_ROOT / "sub-01").rglob("*.edf"))

stats_rows = []
for f in sub01_files:
    try:
        r = mne.io.read_raw_edf(f, preload=True, verbose=False)
        d = r.get_data(picks="eeg") * 1e6
        m = re.search(r"run-(\d+)", f.name)
        stats_rows.append({
            "file": f.name,
            "run": int(m.group(1)) if m else None,
            "n_trials": len(r.annotations),
            "n_samples": d.shape[1],
            "mean_uV": float(np.nanmean(d)),
            "std_uV": float(np.nanstd(d)),
            "min_uV": float(np.nanmin(d)),
            "max_uV": float(np.nanmax(d)),
        })
    except Exception as e:
        stats_rows.append({"file": f.name, "error": repr(e)})

df_stats = pd.DataFrame(stats_rows)
df_stats.head(10)

In [ ]:
df_stats[["std_uV", "min_uV", "max_uV"]].describe()

## 7. Simulating Preprocessor — load_data / load_labels

In [ ]:
CHISCO_EEG_CHANNELS = [
    "T9", "FT9", "FTT9h", "T7", "TP7", "TTP7h", "C5", "FTT7h", "Ft7", "10", "11",
    "FFT7h", "FC5", "FCC5h", "CCP5h", "TPP5h", "P7", "P9", "P11", "PO11", "PO9",
    "PPO7", "P5", "CPP5h", "CP3", "C3", "FC3", "FFC5h", "F5", "F7", "AF7", "AFF5h",
    "F3", "FFC3h", "FCC3h", "C1", "CCP3h", "CPP3h", "P3", "PO3", "POO7", "POO9h",
    "POO11h", "I1", "OI1", "POO3", "PO1", "PPO1", "CPP1h", "CP1", "CCP1h", "FCC1h",
    "FC1", "FFC1h", "F1", "AF3", "FP1", "FPz", "AFz", "Fz", "FCz", "FCCz", "Cz",
    "CPPz", "PPOz", "POz", "POOz", "Oz", "Iz", "I2", "OI2", "POO4", "PO2", "PPO2",
    "CPP2h", "CP2", "CCP2h", "FCC2h", "FC2", "FFC2h", "F2", "AF4", "Fp2", "84",
    "85", "AF8", "AFF6h", "F4", "FFC4h", "FCC4h", "C2", "CCP4h", "CPP4h", "P4",
    "PO4", "POO8", "POO10h", "POO12h", "PO12", "PO10", "PPO8", "P6", "CPP6h",
    "CP4", "C4", "FC4", "FFC6h", "F6", "F8", "110", "111", "FFT8h", "FC6", "FCC6h",
    "CCP6h", "TPP8h", "P8", "P10", "P12", "TTP8h", "C6", "FTT8h", "FT8", "T10",
    "FT10", "FTT10h", "T8", "TP8",
]

print(f"EEG channels: {len(CHISCO_EEG_CHANNELS)}")
print(f"Non-EEG: VEO, HEO, EKG, EMG, Trigger")

In [ ]:
def simulate_load_data(file_path: Path) -> mne.io.BaseRaw:
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    ch_type_map = {
        "VEO": "eog", "HEO": "eog",
        "EKG": "ecg",
        "EMG": "emg",
        "Trigger": "stim",
    }
    raw.set_channel_types({ch: t for ch, t in ch_type_map.items() if ch in raw.ch_names})
    return raw


def simulate_load_labels(
    file_path: Path, raw: mne.io.BaseRaw, run_num: int
) -> mne.Annotations:
    TRIAL_DURATION_S = 10.0

    xlsx_path = CHISCO_ROOT / f"textdataset/split_data_{run_num}.xlsx"
    df_xlsx = pd.read_excel(xlsx_path)
    df_xlsx.columns = ["sentence", "category"]

    with open(CHISCO_ROOT / "json/textmaps.json") as f:
        tmap = json.load(f)

    trigger_onsets = raw.annotations.onset[raw.annotations.description == "100"]
    n = min(len(trigger_onsets), len(df_xlsx))

    descriptions = [
        str(tmap.get(df_xlsx.iloc[i]["sentence"], -1))
        for i in range(n)
    ]

    return mne.Annotations(
        onset=trigger_onsets[:n],
        duration=np.full(n, TRIAL_DURATION_S),
        description=descriptions,
    )


raw_loaded = simulate_load_data(SAMPLE_FILE)
ann_loaded = simulate_load_labels(SAMPLE_FILE, raw_loaded, run_num=1)

print(raw_loaded)
print(f"raw shape: {raw_loaded.get_data().shape}")
print(f"sfreq: {raw_loaded.info['sfreq']}")
print(f"n annotations: {len(ann_loaded)}")
print(f"label counts (class_id): {dict(Counter(ann_loaded.description))}")

In [ ]:
# Scan consistency: n_trials in EDF vs n_rows in xlsx
discrepancies = []

for _, inv_row in df_inventory.iterrows():
    run = inv_row["run"]
    if run is None:
        continue
    xlsx_path = CHISCO_ROOT / f"textdataset/split_data_{run}.xlsx"
    if not xlsx_path.exists():
        discrepancies.append({"subject": inv_row["subject"], "run": run, "issue": "xlsx missing"})
        continue
    try:
        r = mne.io.read_raw_edf(inv_row["path"], preload=False, verbose=False)
        n_edf = len(r.annotations)
        n_xlsx = len(pd.read_excel(xlsx_path))
        if n_edf != n_xlsx:
            discrepancies.append({
                "subject": inv_row["subject"],
                "run": run,
                "n_edf_trials": n_edf,
                "n_xlsx_rows": n_xlsx,
            })
    except Exception as e:
        discrepancies.append({"subject": inv_row["subject"], "run": run, "error": repr(e)})

if discrepancies:
    print(f"Mismatches found ({len(discrepancies)}):")
    print(pd.DataFrame(discrepancies))
else:
    print("No mismatches — all EDF trial counts match their xlsx files.")

## 8. Summary

In [ ]:
total_trials = df_trials["n_trials"].dropna().astype(int).sum()

print("=== CHISCO Dataset Summary ===")
print(f"  Subjects          : {len(subjects)}")
print(f"  Total EDF files   : {len(df_inventory)}")
print(f"  EEG channels      : {len(CHISCO_EEG_CHANNELS)} + VEO/HEO/EKG/EMG/Trigger")
print(f"  Sampling rate     : 1000 Hz")
print(f"  Trial duration    : ~10 s per sentence")
print(f"  Total trials      : {total_trials} (across all subjects x runs)")
print(f"  Unique sentences  : {len(text_map)}")
print(f"  Semantic classes  : {len(class_map)}")
print(f"  Trigger marker    : annotation '100' per trial onset")
print(f"  Text data         : textdataset/split_data_{{1..45}}.xlsx (one per run)")